In [1]:
import os
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split

from torch_geometric.nn import global_mean_pool
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch
from torch_geometric.nn import GCNConv, VGAE

import os
import kagglehub
from kagglehub import KaggleDatasetAdapter

import pandas as pd

from tqdm import tqdm
from tqdm.contrib import tmap
from tqdm.contrib.concurrent import process_map

from torchvision import transforms

from concurrent.futures import ProcessPoolExecutor

from lib.lib import SiameseSignatureDataset

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc, precision_recall_fscore_support
import matplotlib.pyplot as plt
import numpy as np

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

C:\Users\princ\Documents\projects\asrs\machine-learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data Preparation
## prepare data from mallapraveen/signature-matching
## and construct it using data.csv

In [2]:
df = pd.read_csv('data.csv')

def dataset_path():
    path = kagglehub.dataset_download("mallapraveen/signature-matching")
    return os.path.join(path, 'custom\\full')

def transform(**kwargs):
    return transforms.Compose([
        transforms.Grayscale(num_output_channels=kwargs['num_output_channels']),
        transforms.Resize(kwargs['resize']),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])  # For 1 channel grayscale
    ])
    
dataset = SiameseSignatureDataset(
    root_dir=dataset_path(),
    signer_folders=df,
    transform=transform(num_output_channels=1, resize=(64, 64)
))

Loaded 85246 signature images (genuine + forged)


## split the data
### train dataset & validation dataset

In [3]:
total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
print(f"Dataset sizes - Train: {train_size}, Validation: {val_size}")

Dataset sizes - Train: 68196, Validation: 17050


In [4]:
train_dataset[0]

(Data(x=[4096, 5], edge_index=[2, 16128]),
 Data(x=[4069, 5], edge_index=[2, 15938]),
 0)

In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    # num_workers=1 # or set it to 4 in GPU device
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    # num_workers=1 # or set it to 4 in GPU device
)

In [6]:
next(iter(train_loader))

[DataBatch(x=[130216, 5], edge_index=[2, 510724], batch=[130216], ptr=[33]),
 DataBatch(x=[130718, 5], edge_index=[2, 513600], batch=[130718], ptr=[33]),
 tensor([1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1,
         0, 1, 1, 1, 0, 1, 0, 0])]

# Model Preparation

In [7]:
class SiameseVGAE(nn.Module):
    class GNNEncoder(torch.nn.Module):
        def __init__(self, in_channels, hidden_channels, latent_dim):
            super().__init__()
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv_mu = GCNConv(hidden_channels, latent_dim)
            self.conv_logvar = GCNConv(hidden_channels, latent_dim)
        
        def forward(self, x, edge_index):
            x = F.relu(self.conv1(x, edge_index))
            mu = self.conv_mu(x, edge_index)
            logvar = self.conv_logvar(x, edge_index)
            return mu, logvar
    
    def __init__(self, in_channels, hidden_channels, latent_dim):
        super().__init__()
        self.encoder = VGAE(self.GNNEncoder(
            in_channels=in_channels,
            hidden_channels=hidden_channels,
            latent_dim=latent_dim
        ))
        self.embedding_dim = latent_dim
    
    def forward_once(self, x, edge_index, batch, use_mu=False):
        if not use_mu:
            z = self.encoder.encode(x, edge_index)
        else:
            mu, _ = self.encoder.encoder(x, edge_index)
            z = mu
        
        graph_emb = global_mean_pool(z, batch)
        graph_emb = F.normalize(graph_emb, p=2, dim=1)
        return graph_emb
    
    def forward(self, x1, x2, edge_index1, edge_index2, batch1, batch2, use_mu=False):
        emb1 = self.forward_once(x1, edge_index1, batch1, use_mu)
        emb2 = self.forward_once(x2, edge_index2, batch2, use_mu)
        return emb1, emb2
    
    def compute_vgae_loss(self, x, edge_index, pos_edge_index=None):
        """Compute VGAE reconstruction + KL loss for a single graph"""
        z = self.encoder.encode(x, edge_index)
        
        # Use provided pos_edge_index or default to edge_index
        if pos_edge_index is None:
            pos_edge_index = edge_index
            
        recon_loss = self.encoder.recon_loss(z, pos_edge_index)
        kl_loss = self.encoder.kl_loss() / x.size(0)  # Normalize by num nodes
        return recon_loss + kl_loss

In [8]:
w_d = 1e-5
epochs = 500
learning_rate = 1e-3
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [9]:
img1, _, _ = next(iter(train_loader))

input_dim = img1.x.shape[1]
hidden_dim = 64
latent_dim = 128

In [10]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, distance, label):
        # label = 1 → same, label = 0 → different
        loss = label * distance.pow(2) + \
               (1 - label) * F.relu(self.margin - distance).pow(2)
        return loss.mean()

In [11]:
model = SiameseVGAE(in_channels=input_dim, hidden_channels=hidden_dim, latent_dim=latent_dim).to(device)
# criterion = nn.CrossEntropyLoss()
criterion = ContrastiveLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [12]:
def train_step(model, dataloader, criterion, optimizer, device, threshold=1.0, alpha=0.5, beta=0.25):
    """
    alpha: weight for contrastive loss
    beta: weight for VGAE loss (applied to each graph)
    """
    model.train()
    total_loss, total_contrastive_loss, total_vgae_loss = 0.0, 0.0, 0.0
    correct, total = 0, 0
    
    for x1, x2, label in tqdm(dataloader, desc="Training", leave=False):
        # Move to device
        x1, x2, label = x1.to(device), x2.to(device), label.to(device).float()
        
        # Forward: get embeddings (use sampled z during training)
        emb1, emb2 = model(
            x1.x, x2.x,
            x1.edge_index, x2.edge_index,
            x1.batch, x2.batch,
            use_mu=False  # Use stochastic sampling
        )
        
        # Compute distance and contrastive loss
        distance = F.pairwise_distance(emb1, emb2)
        contrastive_loss = criterion(distance, label)
        
        # Compute VGAE losses for both graphs
        vgae_loss1 = model.compute_vgae_loss(x1.x, x1.edge_index)
        vgae_loss2 = model.compute_vgae_loss(x2.x, x2.edge_index)
        vgae_loss = vgae_loss1 + vgae_loss2
        
        # Combined loss
        loss = alpha * contrastive_loss + beta * vgae_loss
        
        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
        preds = (distance < threshold).float()
        correct += (preds == label).sum().item()
        total += label.size(0)
        
        total_loss += loss.item() * label.size(0)
        total_contrastive_loss += contrastive_loss.item() * label.size(0)
        total_vgae_loss += vgae_loss.item() * label.size(0)
    
    avg_loss = total_loss / total
    avg_contrastive_loss = total_contrastive_loss / total
    avg_vgae_loss = total_vgae_loss / total
    accuracy = correct / total
    
    return avg_loss, avg_contrastive_loss, avg_vgae_loss, accuracy


def val_step(model, dataloader, criterion, device, threshold=1.0, alpha=0.5, beta=0.25):
    model.eval()
    total_loss, total_contrastive_loss, total_vgae_loss = 0.0, 0.0, 0.0
    correct, total = 0, 0
    all_labels = []
    all_preds = []
    all_distances = []
    
    with torch.no_grad():
        for x1, x2, label in tqdm(dataloader, desc="Validating", leave=False):
            x1, x2, label = x1.to(device), x2.to(device), label.to(device).float()
            
            # Forward pass: get embeddings (use mu for deterministic evaluation)
            emb1, emb2 = model(
                x1.x, x2.x,
                x1.edge_index, x2.edge_index,
                x1.batch, x2.batch,
                use_mu=True  # Use deterministic mu
            )
            
            # Compute distance and contrastive loss
            distance = F.pairwise_distance(emb1, emb2)
            contrastive_loss = criterion(distance, label)
            
            # Compute VGAE losses
            vgae_loss1 = model.compute_vgae_loss(x1.x, x1.edge_index)
            vgae_loss2 = model.compute_vgae_loss(x2.x, x2.edge_index)
            vgae_loss = vgae_loss1 + vgae_loss2
            
            # Combined loss
            loss = alpha * contrastive_loss + beta * vgae_loss
            
            total_loss += loss.item() * label.size(0)
            total_contrastive_loss += contrastive_loss.item() * label.size(0)
            total_vgae_loss += vgae_loss.item() * label.size(0)
            
            # Compute predictions
            preds = (distance < threshold).float()
            correct += (preds == label).sum().item()
            total += label.size(0)
            
            # Store for metrics
            all_labels.extend(label.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_distances.extend(distance.cpu().numpy())
    
    avg_loss = total_loss / total
    avg_contrastive_loss = total_contrastive_loss / total
    avg_vgae_loss = total_vgae_loss / total
    accuracy = correct / total
    
    return avg_loss, avg_contrastive_loss, avg_vgae_loss, accuracy, np.array(all_labels), np.array(all_preds), np.array(all_distances)

In [ ]:
writer = SummaryWriter(log_dir="runs/siamese_signature_experiment")
patience = 10
best_auc = 0.0
best_val_loss = float('inf')
wait = 0

# Loss weights
alpha = 0.5  # Contrastive loss weight
beta = 0.25  # VGAE loss weight

for epoch in range(epochs):
    train_loss, train_contrastive, train_vgae, train_acc = train_step(
        model, train_loader, criterion, optimizer, device, 
        alpha=alpha, beta=beta
    )
    
    val_loss, val_contrastive, val_vgae, val_acc, y_true, y_pred, y_distance = val_step(
        model, val_loader, criterion, device, threshold=0.5,
        alpha=alpha, beta=beta
    )
    
    # --- ROC Curve and AUC ---
    y_score = -y_distance
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    
    # --- Confusion Matrix ---
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    fig_cm, ax_cm = plt.subplots(figsize=(4, 4))
    disp.plot(ax=ax_cm, cmap="Blues", colorbar=False)
    writer.add_figure("ConfusionMatrix/val", fig_cm, global_step=epoch)
    plt.close(fig_cm)
    
    # --- Precision, Recall, F1 ---
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary"
    )
    
    # Log all metrics including VGAE losses
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/train_contrastive", train_contrastive, epoch)
    writer.add_scalar("Loss/train_vgae", train_vgae, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("Loss/val_contrastive", val_contrastive, epoch)
    writer.add_scalar("Loss/val_vgae", val_vgae, epoch)
    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)
    writer.add_scalar("AUC/val", roc_auc, epoch)
    writer.add_scalar("Precision/val", precision, epoch)
    writer.add_scalar("Recall/val", recall, epoch)
    writer.add_scalar("F1/val", f1, epoch)
    
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f} (C: {train_contrastive:.4f}, V: {train_vgae:.4f}) | Train Acc: {train_acc:.4f} "
          f"| Val Loss: {val_loss:.4f} (C: {val_contrastive:.4f}, V: {val_vgae:.4f}) | Val Acc: {val_acc:.4f} "
          f"| AUC: {roc_auc:.4f} | F1: {f1:.4f}")
    
    # --- Save best model by AUC ---
    if roc_auc > best_auc:
        best_auc = roc_auc
        torch.save(model.state_dict(), "best_siamesevgae_signature.pth")
    
    # --- Early stopping based on validation loss ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        torch.save(model.state_dict(), os.path.join(writer.log_dir, "best_siamesevgae_signature.pth"))
    else:
        wait += 1
        if wait >= patience:
            print(f"⏹️ Early stopping triggered at epoch {epoch+1}!")
            break

writer.close()

Training:   0%|                                                                               | 0/2132 [00:00<?, ?it/s]